__<h1 style="text-align: center;font-size: 3rem">Model Exploration</h1><p style="text-align: center;font-size: 1.3rem">(Notebook III)</p>__

## Imports

Now entering model exploration, classification models which will classify whether a transaction is genuine or fraudulent. These models come primarily from _'Scikit-Learn'_.

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    make_scorer,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
)
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.feature_selection import RFE, SelectKBest, mutual_info_classif, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler, NearMiss
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
from dotenv import load_dotenv
from os import getenv

import pandas as pd
import numpy as np

from functools import partial
import warnings

from fraud_detection_pipeline.utils.constants import (
    PARENT_DIR,
    PROCESSED_DIRECTORY,
    CPU_COUNT,
)
from fraud_detection_pipeline.data_loading import tts_parquet
from fraud_detection_pipeline.evaluations import (
    EvaluationEntry,
    EvaluationResults,
)

## Setup

Loading the random state to be used throughout the notebook and project as a whole.

In [ ]:
load_dotenv()

RANDOM_STATE = int(getenv("RANDOM_STATE", 0))
USE_ALL_CPU_CORES = bool(getenv("USE_ALL_CPUS_CORES", 0))
TEST_SPLIT_SIZE = float(getenv("TEST_SPLIT_SIZE", 0.3))

RANDOM_STATE, USE_ALL_CPU_CORES, TEST_SPLIT_SIZE

(39106, True)

In [3]:
n_jobs = CPU_COUNT if USE_ALL_CPU_CORES else 1

In [4]:
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

`FeatureTarget` is a named tuple that contains the features (`X`) and the targets (`y`) that organizes the way the training and testing data the columns in itself.

Transactions are loaded from the minimally processed parquet file.

In [5]:
X_train, X_test, y_train, y_test = tts_parquet(
    PARENT_DIR / PROCESSED_DIRECTORY / "creditcard.parquet",
    target="is_fraud",
    test_size=0.25,
    random_state=RANDOM_STATE,
)

Separating the numeric features from the categorical features will help with appropriately transforming the data for the use of models. The features of the dataset is continuous so categorical transformations are not necessary.

In [6]:
numeric = X_train.select_dtypes(include=["float64", "int64", "int32"]).columns.tolist()

categorical = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [7]:
print(f"Numeric Features: {numeric}", f"Categorical Features: {categorical}", sep="\n")

Numeric Features: ['v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20', 'v21', 'v22', 'v23', 'v24', 'v25', 'v26', 'v27', 'v28', 'amount', 'hour_sin', 'hour_cos']
Categorical Features: []


In [8]:
def pretty_print_metrics(
    y_true: np.ndarray | pd.Series,
    y_pred: np.ndarray | pd.Series,
):
    cm_df = pd.DataFrame(
        confusion_matrix(y_true, y_pred),
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"],
    )

    print(
        "Confusion Matrix",
        "----------------",
        cm_df,
        "\n",
        "Classification Report",
        "---------------------",
        classification_report(y_true, y_pred, digits=4),
        sep="\n",
    )

In [9]:
preproc = ColumnTransformer(
    [
        (
            "numeric",
            StandardScaler(),
            numeric,
        ),
    ],
    remainder="passthrough",
)

In [10]:
results = EvaluationResults()

## Baseline Model

Using two baseline models, one being completely random and the other being a simple logistic regression, tuned models will be evaluated along side these two baselines to ensure it is doing better than at least complete random selection and a simple model.

### Random baseline model using a Dummy Classifier

In [11]:
stratified_base = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
stratified_base.fit(X_train, y_train)

,strategy,'stratified'
,random_state,39106
,constant,None


The random baseline model's performance on testing data

In [12]:
y_pred = stratified_base.predict(X_test)
pretty_print_metrics(y_test, y_pred)
print(average_precision_score(y_test, y_pred))

result = EvaluationEntry(
    name="stratified_base",
    key=average_precision_score(y_test, y_pred),
    accuracy=accuracy_score(y_test, y_pred),
    estimator=stratified_base,
    params=stratified_base.get_params(),
)
results.append(result)

Confusion Matrix
----------------
          Predicted 0  Predicted 1
Actual 0        70974          122
Actual 1          106            0


Classification Report
---------------------
              precision    recall  f1-score   support

       False     0.9985    0.9983    0.9984     71096
        True     0.0000    0.0000    0.0000       106

    accuracy                         0.9968     71202
   macro avg     0.4993    0.4991    0.4992     71202
weighted avg     0.9970    0.9968    0.9969     71202

0.0014887222269037387


In [13]:
majority_base = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
majority_base.fit(X_train, y_train)

,strategy,'most_frequent'
,random_state,39106
,constant,None


In [14]:
y_pred = majority_base.predict(X_test)
pretty_print_metrics(y_test, y_pred)
print(average_precision_score(y_test, y_pred))

result = EvaluationEntry(
    name="majority_base",
    key=average_precision_score(y_test, y_pred),
    accuracy=accuracy_score(y_test, y_pred),
    estimator=majority_base,
    params=majority_base.get_params(),
)
results.append(result)

Confusion Matrix
----------------
          Predicted 0  Predicted 1
Actual 0        71096            0
Actual 1          106            0


Classification Report
---------------------
              precision    recall  f1-score   support

       False     0.9985    1.0000    0.9993     71096
        True     0.0000    0.0000    0.0000       106

    accuracy                         0.9985     71202
   macro avg     0.4993    0.5000    0.4996     71202
weighted avg     0.9970    0.9985    0.9978     71202

0.0014887222269037387


### Simple baseline using a Logistic Regression

In [15]:
log_base = Pipeline(
    steps=[
        ("preprocessor", preproc),
        (
            "classifier",
            LogisticRegression(
                random_state=RANDOM_STATE,
                max_iter=1500,
            ),
        ),
    ]
)

log_base.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


The simple baseline model's performance on testing data

In [16]:
y_preds = log_base.predict(X_test)
pretty_print_metrics(y_test, y_preds)
print(average_precision_score(y_test, y_preds))

result = EvaluationEntry(
    name="base_logistic",
    key=average_precision_score(y_test, y_preds),
    accuracy=accuracy_score(y_test, y_preds),
    estimator=log_base,
    params=log_base.get_params(),
)
results.append(result)

Confusion Matrix
----------------
          Predicted 0  Predicted 1
Actual 0        71080           16
Actual 1           45           61


Classification Report
---------------------
              precision    recall  f1-score   support

       False     0.9994    0.9998    0.9996     71096
        True     0.7922    0.5755    0.6667       106

    accuracy                         0.9991     71202
   macro avg     0.8958    0.7876    0.8331     71202
weighted avg     0.9991    0.9991    0.9991     71202

0.45652516815930183


__Notes__

The model are that it's accuracy is extremely high again, attributed to the extreme imbalance in the dataset. Accuracy in this scenario is misleading so the precision and recall are highlighted to understand how the model is handling the imbalance and how sensitive it is to it.

It does perform better than the random baseline, indicating there is a the ability to decern trends in the data. but due to the imbalance, more care will have to be given in the sampling techniques used.

## Resampling the Training Data

In [17]:
num_fraud = int(y_train.sum())
num_genuine = int(y_train.count() - num_fraud)

## Feature Section and Cross Validation Folds

In [18]:
rfe = RFE(
    LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1500,
        fit_intercept=True,
        class_weight="balanced",
        penalty="l1",
    ),
    n_features_to_select=10,
)

skb = SelectKBest(mutual_info_classif, k=10)

In [19]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

In [20]:
for k, (_, test_index) in enumerate(cv.split(X_train, y_train)):
    counts = np.bincount(y_train.iloc[test_index].astype(int), minlength=2)
    print(f"Fold {k}: neg={counts[0]}, pos={counts[1]}")

Fold 0: neg=71073, pos=129
Fold 1: neg=71073, pos=129
Fold 2: neg=71073, pos=128


## Model Candidates and Grid Search Parameters

The models considered are models that can handle primarily continuous numbers. The models considered are as follows

- Logistic Classifier with L1 Penalty (`LogisticRegression("l1", ...)`)
- Logistic Classifier with L2 Penalty (`LogisiticRegression("l2", ...)`)
- Logistic Classifier with ElasticNet Penalty (`LogisiticRegression("elasticnet", ...)`)
- Decision Tree Classifier (`DecisionTreeClassifier(...)`)
- Random Forest Classifier(`RandomForestClassifer(...)`)
- kNN Classifier (`KNeighborsClassifier(...)`)
- Stochastic Gradient Descent Classifier with L1 Penalty (`SGDClassifier("l1", ...)`)
- Stochastic Gradient Descent Classifier with L2 Penalty (`SGDClassifier("l2", ...)`)
- XGBoost Classifier (`XGBClassifier`)

In [21]:
estimator = Pipeline(
    steps=[
        ("preprocessor", "passthrough"),
        ("over_sampler", "passthrough"),
        ("under_sampler", "passthrough"),
        ("feature_selection", "passthrough"),
        ("classifier", "passthrough"),
    ]
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(precision_score),
    "recall": make_scorer(recall_score),
    "f1": make_scorer(f1_score),
    "roc_auc": make_scorer(roc_auc_score),
    "average_precision": make_scorer(average_precision_score),
}

refit = "average_precision"

make_grid = partial(
    GridSearchCV,
    estimator,
    cv=cv,
    scoring=scoring,
    n_jobs=n_jobs,
    verbose=1,
    refit=refit,
)

### Logisitic Regression with L1 Penalty

With a L1 Penalty, coefficients of the model are penalized to extremely low numbers or zero. This provides integrated feature selection in the model as only the best make it into the estimator.

In [22]:
l1_log_params = {
    "preprocessor": [preproc],
    "classifier": [
        LogisticRegression(
            penalty="l1",
            tol=1e-3,
            random_state=RANDOM_STATE,
            max_iter=12500,
            fit_intercept=True,
        )
    ],
    "classifier__C": [0.001, 0.01, 0.1, 1],
    "classifier__solver": ["saga", "liblinear"],
    "classifier__penalty": ["l1"],
    "classifier__class_weight": [None, {0: 1, 1: 3}, {0: 1, 1: 5}],
}

In [23]:
l1_log_search = make_grid(l1_log_params)
l1_log_search.fit(X_train, y_train)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


,estimator,Pipeline(step...assthrough')])
,param_grid,"{'classifier': [LogisticRegre...06, tol=0.001)], 'classifier__C': [0.001, 0.01, ...], 'classifier__class_weight': [None, {0: 1, 1: 3}, ...], 'classifier__penalty': ['l1'], ...}"
,scoring,"{'accuracy': make_scorer(a...hod='predict'), 'average_precision': make_scorer(a...hod='predict'), 'f1': make_scorer(f...hod='predict'), 'precision': make_scorer(p...hod='predict'), ...}"
,n_jobs,12
,refit,'average_precision'
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric', ...)]"


In [24]:
preds = l1_log_search.predict(X_test)
pretty_print_metrics(y_test, preds)
print(average_precision_score(y_test, preds))

results.append(
    EvaluationEntry(
        name="l1_log",
        key=average_precision_score(y_test, preds),
        accuracy=accuracy_score(y_test, preds),
        estimator=l1_log_search.best_estimator_,
        params=l1_log_search.best_params_,
    )
)

Confusion Matrix
----------------
          Predicted 0  Predicted 1
Actual 0        71068           28
Actual 1           22           84


Classification Report
---------------------
              precision    recall  f1-score   support

       False     0.9997    0.9996    0.9996     71096
        True     0.7500    0.7925    0.7706       106

    accuracy                         0.9993     71202
   macro avg     0.8748    0.8960    0.8851     71202
weighted avg     0.9993    0.9993    0.9993     71202

0.5946486027263385


In [25]:
l1_log_rs_params = {
    "preprocessor": [preproc],
    "over_sampler": [SMOTE(sampling_strategy=0.0034, random_state=RANDOM_STATE)],
    "over_sampler__k_neighbors": [3, 5, 7],
    "classifier": [
        LogisticRegression(
            penalty="l1",
            tol=1e-3,
            random_state=RANDOM_STATE,
            max_iter=12500,
            fit_intercept=True,
        )
    ],
    "classifier__C": [0.001, 0.01, 0.1, 1],
    "classifier__solver": ["saga", "liblinear"],
    "classifier__class_weight": [None, {0: 1, 1: 3}, {0: 1, 1: 5}],
}

In [26]:
l1_log_rs_search = make_grid(l1_log_rs_params)
l1_log_rs_search.fit(X_train, y_train)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


,estimator,Pipeline(step...assthrough')])
,param_grid,"{'classifier': [LogisticRegre...06, tol=0.001)], 'classifier__C': [0.001, 0.01, ...], 'classifier__class_weight': [None, {0: 1, 1: 3}, ...], 'classifier__solver': ['saga', 'liblinear'], ...}"
,scoring,"{'accuracy': make_scorer(a...hod='predict'), 'average_precision': make_scorer(a...hod='predict'), 'f1': make_scorer(f...hod='predict'), 'precision': make_scorer(p...hod='predict'), ...}"
,n_jobs,12
,refit,'average_precision'
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric', ...)]"


In [27]:
preds = l1_log_rs_search.predict(X_test)
pretty_print_metrics(y_test, preds)
print(average_precision_score(y_test, preds))

results.append(
    EvaluationEntry(
        name="l1_log_rs",
        key=average_precision_score(y_test, preds),
        accuracy=accuracy_score(y_test, preds),
        estimator=l1_log_rs_search.best_estimator_,
        params=l1_log_rs_search.best_params_,
    )
)

Confusion Matrix
----------------
          Predicted 0  Predicted 1
Actual 0        71069           27
Actual 1           21           85


Classification Report
---------------------
              precision    recall  f1-score   support

       False     0.9997    0.9996    0.9997     71096
        True     0.7589    0.8019    0.7798       106

    accuracy                         0.9993     71202
   macro avg     0.8793    0.9008    0.8897     71202
weighted avg     0.9993    0.9993    0.9993     71202

0.6088697333791844


### Logistic Regression with L2 Penalty

Using L2 Penalty avoids multicolinearity to reduce overfitting, still retaining most features and not diminishing them.

In [28]:
l2_log_params = {
    "preprocessor": [preproc],
    # "over_sampler": [SMOTE(random_state=RANDOM_STATE)],
    # "over_sampler__k_neighbors": [3, 5, 7],
    # "over_sampler__sampling_strategy": [{1: num_fraud * 2}],
    # "under_sampler": [NearMiss()],
    # "under_sampler__sampling_strategy": [1 / 14],
    "feature_selection": [SelectKBest()],
    "feature_selection__score_func": [mutual_info_classif, f_classif],
    "feature_selection__k": [10, 15, 20, 25, "all"],
    "classifier": [
        LogisticRegression(
            penalty="l2",
            tol=1e-3,
            random_state=RANDOM_STATE,
            max_iter=12500,
            fit_intercept=True,
        )
    ],
    "classifier__C": [0.001, 0.01, 0.1, 1],
    "classifier__solver": ["lbfgs", "sag", "saga"],
    "classifier__class_weight": [None, {0: 1, 1: 3}, {0: 1, 1: 5}],
}

In [29]:
l2_log_search = make_grid(l2_log_params)
l2_log_search.fit(X_train, y_train)

Fitting 3 folds for each of 360 candidates, totalling 1080 fits


,estimator,Pipeline(step...assthrough')])
,param_grid,"{'classifier': [LogisticRegre...06, tol=0.001)], 'classifier__C': [0.001, 0.01, ...], 'classifier__class_weight': [None, {0: 1, 1: 3}, ...], 'classifier__solver': ['lbfgs', 'sag', ...], ...}"
,scoring,"{'accuracy': make_scorer(a...hod='predict'), 'average_precision': make_scorer(a...hod='predict'), 'f1': make_scorer(f...hod='predict'), 'precision': make_scorer(p...hod='predict'), ...}"
,n_jobs,12
,refit,'average_precision'
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric', ...)]"


In [30]:
preds = l2_log_search.predict(X_test)
pretty_print_metrics(y_test, preds)
print(average_precision_score(y_test, preds))

result = EvaluationEntry(
    name="l2_log",
    key=average_precision_score(y_test, preds),
    accuracy=accuracy_score(y_test, preds),
    estimator=l2_log_search.best_estimator_,
    params=l2_log_search.best_params_,
)
results.append(result)

Confusion Matrix
----------------
          Predicted 0  Predicted 1
Actual 0        71074           22
Actual 1           24           82


Classification Report
---------------------
              precision    recall  f1-score   support

       False     0.9997    0.9997    0.9997     71096
        True     0.7885    0.7736    0.7810       106

    accuracy                         0.9994     71202
   macro avg     0.8941    0.8866    0.8903     71202
weighted avg     0.9993    0.9994    0.9994     71202

0.6102790140310551


In [ ]:
elasticnet_log_params = {
    "preprocessor": [preproc],
    # "feature_selection": [skb],
    # "feature_selection__k": [10, 15, 20, "all"],
    "classifier": [
        LogisticRegression(
            penalty="elasticnet",
            tol=1e-3,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            max_iter=12500,
            fit_intercept=True,
        )
    ],
    "classifier__C": [0.001, 0.01, 0.1, 1],
    "classifier__solver": ["liblinear", "sag", "saga"],
    "classifier__l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0],
    "classifier__class_weight": [None, {0: 1, 1: 3}, {0: 1, 1: 5}],
}

In [32]:
elasticnet_log_search = make_grid(elasticnet_log_params)
elasticnet_log_search.fit(X_train, y_train)

Fitting 3 folds for each of 180 candidates, totalling 540 fits


/home/jrhol/programming/projects/fraud-detection-pipeline/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 540.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/jrhol/programming/projects/fraud-detection-pipeline/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/jrhol/programming/projects/fraud-detection-pipeline/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^

,estimator,Pipeline(step...assthrough')])
,param_grid,"{'classifier': [LogisticRegre...06, tol=0.001)], 'classifier__C': [0.001, 0.01, ...], 'classifier__class_weight': [None, {0: 1, 1: 3}, ...], 'classifier__l1_ratio': [0.0, 0.25, ...], ...}"
,scoring,"{'accuracy': make_scorer(a...hod='predict'), 'average_precision': make_scorer(a...hod='predict'), 'f1': make_scorer(f...hod='predict'), 'precision': make_scorer(p...hod='predict'), ...}"
,n_jobs,12
,refit,'average_precision'
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric', ...)]"


In [33]:
preds = elasticnet_log_search.predict(X_test)
pretty_print_metrics(y_test, preds)
print(average_precision_score(y_test, preds))

result = EvaluationEntry(
    name="elasticnet_log",
    key=average_precision_score(y_test, preds),
    accuracy=accuracy_score(y_test, preds),
    estimator=elasticnet_log_search.best_estimator_,
    params=elasticnet_log_search.best_params_,
)
results.append(result)

Confusion Matrix
----------------
          Predicted 0  Predicted 1
Actual 0        71068           28
Actual 1           22           84


Classification Report
---------------------
              precision    recall  f1-score   support

       False     0.9997    0.9996    0.9996     71096
        True     0.7500    0.7925    0.7706       106

    accuracy                         0.9993     71202
   macro avg     0.8748    0.8960    0.8851     71202
weighted avg     0.9993    0.9993    0.9993     71202

0.5946486027263385


In [34]:
l1_sgd_params = {
    "preprocessor": [preproc],
    # "feature_selection": [skb],
    # "feature_selection__k": [10, 15, 20, "all"],
    "classifier": [
        SGDClassifier(
            penalty="l1",
            random_state=RANDOM_STATE,
            max_iter=12500,
            fit_intercept=True,
        )
    ],
    "classifier__loss": ["log_loss", "hinge"],
    "classifier__alpha": [1e-4, 1e-3, 1e-2, 1e-1],
    "classifier__class_weight": [None, {0: 1, 1: 3}, {0: 1, 1: 5}],
    "classifier__learning_rate": ["optimal", "adaptive"],
    "classifier__eta0": [1e-3, 1e-2, 1e-1],
}

In [35]:
l1_sgd_search = make_grid(l1_sgd_params)
l1_sgd_search.fit(X_train, y_train)

Fitting 3 folds for each of 144 candidates, totalling 432 fits


,estimator,Pipeline(step...assthrough')])
,param_grid,"{'classifier': [SGDClassifier...m_state=39106)], 'classifier__alpha': [0.0001, 0.001, ...], 'classifier__class_weight': [None, {0: 1, 1: 3}, ...], 'classifier__eta0': [0.001, 0.01, ...], ...}"
,scoring,"{'accuracy': make_scorer(a...hod='predict'), 'average_precision': make_scorer(a...hod='predict'), 'f1': make_scorer(f...hod='predict'), 'precision': make_scorer(p...hod='predict'), ...}"
,n_jobs,12
,refit,'average_precision'
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric', ...)]"


In [36]:
preds = l1_sgd_search.predict(X_test)
pretty_print_metrics(y_test, preds)
print(average_precision_score(y_test, preds))

result = EvaluationEntry(
    name="l1_sgd",
    key=average_precision_score(y_test, preds),
    accuracy=accuracy_score(y_test, preds),
    estimator=l1_sgd_search.best_estimator_,
    params=l1_sgd_search.best_params_,
)
results.append(result)

Confusion Matrix
----------------
          Predicted 0  Predicted 1
Actual 0        71077           19
Actual 1           22           84


Classification Report
---------------------
              precision    recall  f1-score   support

       False     0.9997    0.9997    0.9997     71096
        True     0.8155    0.7925    0.8038       106

    accuracy                         0.9994     71202
   macro avg     0.9076    0.8961    0.9018     71202
weighted avg     0.9994    0.9994    0.9994     71202

0.6465811911124898


In [37]:
l2_sgd_params = {
    "preprocessor": [preproc],
    "feature_selection": [skb],
    "feature_selection__k": [10, 15, 20, "all"],
    "classifier": [
        SGDClassifier(
            penalty="l2",
            random_state=RANDOM_STATE,
            max_iter=12500,
            fit_intercept=True,
        )
    ],
    "classifier__loss": ["log_loss", "hinge"],
    "classifier__alpha": [1e-4, 1e-3, 1e-2, 1e-1],
    "classifier__class_weight": [None, {0: 1, 1: 3}, {0: 1, 1: 5}],
    "classifier__learning_rate": ["optimal", "adaptive"],
    "classifier__eta0": [1e-3, 1e-2, 1e-1],
    "classifier__l1_ratio": [0.1, 0.5, 0.9],
}

In [38]:
l2_sgd_search = make_grid(l2_sgd_params)
l2_sgd_search.fit(X_train, y_train)

Fitting 3 folds for each of 1728 candidates, totalling 5184 fits


KeyboardInterrupt: 

In [ ]:
preds = l2_sgd_search.predict(X_test)
pretty_print_metrics(y_test, preds)
print(average_precision_score(y_test, preds))

result = EvaluationEntry(
    name="l2_sgd",
    key=average_precision_score(y_test, preds),
    accuracy=accuracy_score(y_test, preds),
    estimator=l2_sgd_search.best_estimator_,
    params=l2_sgd_search.best_params_,
)
results.append(result)

In [ ]:
elasticnet_sgd_params = {
    "preprocessor": [preproc],
    # "feature_selection": [skb],
    # "feature_selection__k": [10, 15, 20, "all"],
    "classifier": [
        SGDClassifier(
            penalty="elasticnet",
            random_state=RANDOM_STATE,
            max_iter=12500,
            fit_intercept=True,
            n_jobs=n_jobs,
        )
    ],
    "classifier__loss": ["log_loss", "hinge"],
    "classifier__alpha": [1e-4, 1e-3, 1e-2, 1e-1],
    "classifier__class_weight": [None, {0: 1, 1: 3}, {0: 1, 1: 5}],
    "classifier__learning_rate": ["optimal", "adaptive"],
    "classifier__eta0": [1e-3, 1e-2, 1e-1],
}

In [ ]:
elasticnet_sgd_search = make_grid(elasticnet_sgd_params)
elasticnet_sgd_search.fit(X_train, y_train)

In [ ]:
preds = elasticnet_sgd_search.predict(X_test)
pretty_print_metrics(y_test, preds)
print(average_precision_score(y_test, preds))

result = EvaluationEntry(
    name="elasticnet_sgd",
    key=average_precision_score(y_test, preds),
    accuracy=accuracy_score(y_test, preds),
    estimator=elasticnet_sgd_search.best_estimator_,
    params=elasticnet_sgd_search.best_params_,
)
results.append(result)

In [ ]:
rf_params = {
    "preprocessor": [preproc],
    "classifier": [
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=n_jobs,
        )
    ],
    "classifier__max_depth": [10, 20, 30, None],
    "classifier__min_samples_leaf": [1, 2, 5],
    "classifier__max_features": ["sqrt", "log2", None],
    "classifier__class_weight": [None, "balanced"],
}

In [ ]:
rf_search = make_grid(rf_params)
rf_search.fit(X_train, y_train)

Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV] END classifier=RandomForestClassifier(min_samples_leaf=2, min_samples_split=5,
                       random_state=39106), classifier__class_weight=None, classifier__max_depth=10, classifier__max_features=log2, classifier__min_samples_leaf=1, preprocessor=ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric', StandardScaler(),
                                 ['v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7',
                                  'v8', 'v9', 'v10', 'v11', 'v12', 'v13', 'v14',
                                  'v15', 'v16', 'v17', 'v18', 'v19', 'v20',
                                  'v21', 'v22', 'v23', 'v24', 'v25', 'v26',
                                  'v27', 'v28', 'amount', 'hour_sin', ...])]); total time= 1.1min
[CV] END classifier=RandomForestClassifier(min_samples_leaf=2, min_samples_split=5,
                       random_state=39106), classifier__class_weight=None, classi

,estimator,Pipeline(step...assthrough')])
,param_grid,"{'classifier': [RandomForestC...m_state=39106)], 'classifier__class_weight': [None, 'balanced'], 'classifier__max_depth': [10, 20, ...], 'classifier__max_features': ['sqrt', 'log2', ...], ...}"
,scoring,"{'accuracy': make_scorer(a...hod='predict'), 'average_precision': make_scorer(a...hod='predict'), 'f1': make_scorer(f...hod='predict'), 'precision': make_scorer(p...hod='predict'), ...}"
,n_jobs,12
,refit,'average_precision'
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric', ...)]"


In [ ]:
preds = rf_search.predict(X_test)
pretty_print_metrics(y_test, preds)
print(average_precision_score(y_test, preds))

result = EvaluationEntry(
    name="rf",
    key=average_precision_score(y_test, preds),
    accuracy=accuracy_score(y_test, preds),
    estimator=rf_search.best_estimator_,
    params=rf_search.best_params_,
)
results.append(result)

Confusion Matrix
----------------
          Predicted 0  Predicted 1
Actual 0        71089            7
Actual 1           27           79


Classification Report
---------------------
              precision    recall  f1-score   support

       False     0.9996    0.9999    0.9998     71096
        True     0.9186    0.7453    0.8229       106

    accuracy                         0.9995     71202
   macro avg     0.9591    0.8726    0.9113     71202
weighted avg     0.9995    0.9995    0.9995     71202

0.6849996503961024


In [ ]:
xgboost_params = {
    "preprocessor": [preproc],
    "classifier": [
        XGBClassifier(
            seed=RANDOM_STATE,
            scale_pos_weight=num_genuine / num_fraud,
            objective="binary:logistic",
            subsample=0.8,
        )
    ],
    "classifier__learning_rate": [0.01, 0.1, 0.2],
    "classifier__n_estimators": [100, 200, 400, 600],
    "classifier__max_leaves": [3, 5, 9],
    "classifier__alpha": [0, 0.1, 1, 10],
    "classifier__lambda": [0, 0.1, 1, 10],
    "classifier__eval_metric": ["auc", "aucpr"],
    "classifier__eta": [0.01, 0.1, 0.2],
}

In [ ]:
xgboost_search = make_grid(xgboost_params)
xgboost_search.fit(X_train, y_train)

In [ ]:
preds = xgboost_search.predict(X_test)
pretty_print_metrics(y_test, preds)
print(average_precision_score(y_test, preds))

result = EvaluationEntry(
    name="xgboost",
    key=average_precision_score(y_test, preds),
    accuracy=accuracy_score(y_test, preds),
    estimator=xgboost_search.best_estimator_,
    params=xgboost_search.best_params_,
)
results.append(result)

In [ ]:
results.report_best()

[EvaluationEntry(name='rf', key=0.6849996503961024, accuracy=0.999522485323446, estimator=Pipeline(steps=[('preprocessor',
                  ColumnTransformer(remainder='passthrough',
                                    transformers=[('numeric', StandardScaler(),
                                                   ['v1', 'v2', 'v3', 'v4', 'v5',
                                                    'v6', 'v7', 'v8', 'v9',
                                                    'v10', 'v11', 'v12', 'v13',
                                                    'v14', 'v15', 'v16', 'v17',
                                                    'v18', 'v19', 'v20', 'v21',
                                                    'v22', 'v23', 'v24', 'v25',
                                                    'v26', 'v27', 'v28',
                                                    'amount', 'hour_sin', ...])])),
                 ('over_sampler', 'passthrough'),
                 ('under_sampler', 'passthrough'),
